# **Подготовка данных**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# Скачивание и распаковка датасета
!wget https://storage.yandexcloud.net/aiueducation/Content/base/l4/bus.zip -q
!unzip -q bus.zip -d bus_dataset

# Путь к папкам
base_dir = 'bus_dataset'
categories = ['Входящий', 'Выходящий']
img_size = 64 # Размер, к которому приведем все фото

# **Загрузка и обработка изображений**

In [ ]:
def load_data():
    data = []
    labels = []

    for category in categories:
        path = os.path.join(base_dir, category)
        class_num = categories.index(category) # 0 для 'entering', 1 для 'exiting'

        for img_name in os.listdir(path):
            try:
                # Открываем, меняем размер и нормализуем (делим на 255)
                img = Image.open(os.path.join(path, img_name)).resize((img_size, img_size))
                data.append(np.array(img))
                labels.append(class_num)
            except Exception as e:
                pass

    return np.array(data) / 255.0, np.array(labels)

# Загружаем данные
X, y = load_data()

# Разделяем на обучающую и проверочную выборки
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Размер обучающей выборки: {X_train.shape}')
print(f'Размер проверочной выборки: {X_val.shape}')

Размер обучающей выборки: (7264, 64, 64, 3)
Размер проверочной выборки: (1817, 64, 64, 3)


# **Создание модели**

In [ ]:
model = Sequential([
    # Слой Flatten просто "вытягивает" картинку в один длинный вектор
    Flatten(input_shape=(img_size, img_size, 3)),

    # Первый скрытый слой с большим количеством нейронов
    Dense(512, activation='relu'),
    Dropout(0.3),

    # Второй скрытый слой
    Dense(256, activation='relu'),
    Dropout(0.2),

    # Третий скрытый слой
    Dense(128, activation='relu'),

    # Выходной слой: 1 нейрон с функцией sigmoid (идеально для 2-х классов)
    # 0 - входящий, 1 - выходящий
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 12288)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     6,291,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,456,321 (24.63 MB)

 Trainable params: 6,456,321 (24.63 MB)

 Non-trainable params: 0 (0.00 B)

# **Обучение и результат**

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

# Проверка финальной точности
val_acc = history.history['val_accuracy'][-1]
print(f'\nИтоговая точность на проверочной выборке: {val_acc*100:.2f}%')

Epoch 1/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7032 - loss: 0.6026 - val_accuracy: 0.7325 - val_loss: 0.5301
Epoch 2/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7344 - loss: 0.5466 - val_accuracy: 0.7644 - val_loss: 0.4940
Epoch 3/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7639 - loss: 0.5028 - val_accuracy: 0.7799 - val_loss: 0.4577
Epoch 4/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7876 - loss: 0.4544 - val_accuracy: 0.7887 - val_loss: 0.4520
Epoch 5/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8126 - loss: 0.4099 - val_accuracy: 0.8228 - val_loss: 0.3850
Epoch 6/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8293 - loss: 0.3839 - val_accuracy: 0.8266 - val_loss: 0.3680
Epoch 7/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8400 - loss: 0.3650 - val_accuracy: 0.8277 - val_loss: 0.3684
Epoch 8/50
227/227 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8519 - loss: 0.3325 - val_accuracy: 0.